# TOTO 1.0 — Load


In [ ]:
# !pip install -r requirements/toto.txt


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

REPO_ROOT=Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT/"requirements").exists():
    REPO_ROOT=REPO_ROOT.parent
ROOT=REPO_ROOT/"load"
if str(REPO_ROOT) not in sys.path: sys.path.insert(0,str(REPO_ROOT))
if str(ROOT/"src") not in sys.path: sys.path.insert(0,str(ROOT/"src"))

import os, torch
from toto.data.util.dataset import MaskedTimeseries
from toto.inference.forecaster import TotoForecaster
from toto.model.toto import Toto
from preprocessing import load_data, describe_data, SITES, CONTEXT_LENGTH, PREDICTION_LENGTH
from src.metrics import print_metrics

BASE_QUANTILES=[0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9]
TAIL_QUANTILES=[0.01,0.025,0.05,0.95,0.975,0.99]
OUTPUT_QUANTILES=sorted(set(BASE_QUANTILES+TAIL_QUANTILES))
QUANTILES=BASE_QUANTILES
MODEL_QUANTILES=OUTPUT_QUANTILES
OUTPUT_DIR=ROOT/"outputs"/"predictions"
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
data=load_data(ROOT/"data")
print(describe_data(data))

VARIANT="with_covariates"
FUTURE_COLS=data.covariates
PAST_COLS=[]
DEVICE="cuda" if torch.cuda.is_available() else "cpu"
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG",":4096:8")
toto=Toto.from_pretrained("Datadog/Toto-Open-Base-1.0").to(DEVICE)
forecaster=TotoForecaster(toto.model)


## GEFCom2014-L


In [ ]:
# GEFCom2014-L (cleaned input; no extra cleaning)
from gefcom import load_gefcom_load, load_initial_and_tasks, evaluate_gefcom_load, LOAD_WEATHER
gef_load=load_gefcom_load(ROOT/"data"/"gefcom_load_data.csv")
gef_context, gef_tasks=load_initial_and_tasks(gef_load)
GEF_OUTPUT=ROOT/"outputs"/"predictions"; GEF_OUTPUT.mkdir(parents=True,exist_ok=True)


In [ ]:
def _gef_make_input(context, features, n_exog=0):
    x=torch.tensor(context[features].to_numpy().T,dtype=torch.float32,device=DEVICE)
    n_var,n=x.shape
    kw=dict(series=x,padding_mask=torch.ones_like(x,dtype=torch.bool),id_mask=torch.zeros_like(x),timestamp_seconds=torch.zeros((n_var,n),device=DEVICE),time_interval_seconds=torch.ones(n_var,device=DEVICE))
    if n_exog: kw["num_exogenous_variables"]=n_exog
    return MaskedTimeseries(**kw)

GEF_VARIANT="with_covariates"; rolling=gef_context.copy(); frames=[]
for task_id,future in tqdm(gef_tasks):
    context=rolling.sort_values("timestamp").iloc[-CONTEXT_LENGTH:].copy(); exog=LOAD_WEATHER if GEF_VARIANT=="with_covariates" else []
    inp=_gef_make_input(context,["target",*exog],0)  # weather is historical/past-only for GEFCom2014-L
    H=len(future); fc=forecaster.forecast(inp,prediction_length=H,num_samples=100,use_kv_cache=True); samples=fc.samples[0,0]
    d=pd.DataFrame({"item_id":future.item_id.to_numpy(),"timestamp":future.timestamp.to_numpy(),"task_id":task_id,"prediction":torch.quantile(samples,0.5,dim=-1).cpu().numpy()})
    for q in QUANTILES: d[str(q)]=torch.quantile(samples,q,dim=-1).cpu().numpy()
    frames.append(d); rolling=pd.concat([rolling,future],ignore_index=True)
gef_result=pd.concat(frames,ignore_index=True).merge(gef_load[["item_id","timestamp","target","task_id"]],on=["item_id","timestamp","task_id"],how="inner")
gef_result.to_csv(GEF_OUTPUT/f"toto_gefcom_load_{GEF_VARIANT}.csv",index=False)
gef_summary=evaluate_gefcom_load(gef_result,QUANTILES); display(gef_summary[["task_id","raw_aql","normalized_aql"]])
gef_summary.to_csv(GEF_OUTPUT/f"toto_gefcom_load_summary.csv",index=False)


## PG&E competition


In [ ]:
def make_input(context, features, n_exog=0):
    x=torch.tensor(context[features].to_numpy().T,dtype=torch.float32,device=DEVICE)
    n_var,n=x.shape
    kw=dict(
        series=x,
        padding_mask=torch.ones_like(x,dtype=torch.bool),
        id_mask=torch.zeros_like(x),
        timestamp_seconds=torch.zeros((n_var,n),device=DEVICE),
        time_interval_seconds=torch.ones(n_var,device=DEVICE),
    )
    if n_exog:
        kw["num_exogenous_variables"]=n_exog
    return MaskedTimeseries(**kw)

rolling={s:data.context_by_site[s].copy() for s in data.context_by_site}
features=["target",*PAST_COLS,*FUTURE_COLS] if VARIANT=="with_covariates" else ["target"]
frames=[]

for i in tqdm(range(len(next(iter(data.daily_by_site.values()))))):
    day={
        s:data.daily_by_site[s][i][1].sort_values("timestamp").iloc[:PREDICTION_LENGTH].copy()
        for s in data.daily_by_site
    }

    for site,future in day.items():
        context=rolling[site].sort_values("timestamp").iloc[-CONTEXT_LENGTH:].copy()
        n_exog=len(FUTURE_COLS) if VARIANT=="with_covariates" else 0
        inp=make_input(context,features,n_exog)

        if n_exog:
            future_exog=torch.tensor(
                future[FUTURE_COLS].to_numpy().T,
                dtype=torch.float32,
                device=DEVICE,
            )
            fc=forecaster.forecast(
                inp,
                prediction_length=PREDICTION_LENGTH,
                num_samples=100,
                use_kv_cache=True,
                future_exogenous_variables=future_exog,
            )
        else:
            fc=forecaster.forecast(
                inp,
                prediction_length=PREDICTION_LENGTH,
                num_samples=100,
                use_kv_cache=True,
            )

        samples=fc.samples[0,0]
        qhat={q:torch.quantile(samples,q,dim=-1).detach().cpu().numpy() for q in MODEL_QUANTILES}
        frame=pd.DataFrame({"item_id":future.item_id.iloc[0],"timestamp":future.timestamp.to_numpy(),"prediction":qhat[0.5]})
        for q in MODEL_QUANTILES:
            frame[str(q)]=qhat[q]
        frames.append(frame)

    for site in day:
        rolling[site]=(
            pd.concat([rolling[site],day[site]],ignore_index=True)
            .sort_values("timestamp")
            .iloc[-CONTEXT_LENGTH:]
            .copy()
        )

result=(
    pd.concat(frames,ignore_index=True)
    .merge(data.test[["item_id","timestamp","target"]],on=["item_id","timestamp"],how="inner")
)

# Keep a common 15-quantile CSV schema. Unsupported model quantiles remain NaN.
for _q in OUTPUT_QUANTILES:
    if str(_q) not in result.columns:
        result[str(_q)] = np.nan
result.to_csv(OUTPUT_DIR/f"toto_{VARIANT}.csv",index=False)
print_metrics(result,QUANTILES,label=f"toto_{VARIANT}")
